In [18]:
import pandas as pd
import numpy as np
import sys
import os
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

sys.path.insert(0, '.')  # Add current directory
sys.path.insert(0, '..')  # Add parent directory for clustering.config

# Force fresh import of modules
if 'models_helpers' in sys.modules:
    del sys.modules['models_helpers']
if 'config_models' in sys.modules:
    del sys.modules['config_models']
if 'clustering.config' in sys.modules:
    del sys.modules['clustering.config']

from models_helpers import (
    build_predictor_features,
    clean_measure_values,
    fill_dummy_measure_values,
    is_binary_column,
)
from config_models import TEST_DATA_PATH, TRAIN_DATA_PATH

## 0. Read WP4 Data

In [19]:
df = pd.read_csv("../../output/dataset_wp4.csv.gz", compression='gzip')

## 1. Derive Features

In [20]:
df_features = build_predictor_features(df)
print ((df_features.columns))

Index(['patient_id', 'age', 'cat_household_size', 'sex', 'ethnicity_cat',
       'imd_quintile', 'region', 'rural_urban', 'cat_diabetes', 'smoking',
       'sysbp_value', 'diasbp_value', 'bmi_value',
       'last_hdl_cholesterol_value', 'last_cholesterol_value',
       'last_hba1c_value', 'copd', 'hypertension', 'af', 'ihd', 'ckd',
       'obesity', 'bp_treatment', 'diabetes_treatment', 'mltc_count',
       'has_mltc', 'learndis', 'carehome_at_index', 'housebound', 'smi',
       'homeless', 'substance_abuse', 'migrant', 'non_english_speaking',
       'n_underserved', 'any_underserved', 'asthma_review', 'copd_review',
       'med_review', 'ed_attendances_pre_0_3m', 'ed_attendances_pre_3_6m',
       'ed_attendances_pre_6_9m', 'ed_attendances_pre_9_12m',
       'primary_care_attendances_pre_0_3m',
       'primary_care_attendances_pre_3_6m',
       'primary_care_attendances_pre_6_9m',
       'primary_care_attendances_pre_9_12m', 'hospital_admissions_pre_0_3m',
       'hospital_admissions_p

## 2. Fix Dummy Data

In [21]:
IS_DUMMY_DATA = os.getenv("OPENSAFELY_BACKEND") is None
if IS_DUMMY_DATA:

    df_features = fill_dummy_measure_values(
        df_features,
        missing_prop=0.2,
        seed=42
    )

In [22]:
print(df_features.head())

   patient_id    age cat_household_size     sex ethnicity_cat imd_quintile  \
0          11   76.0            unknown  female         Other      unknown   
1          12  111.0            unknown  female         White      unknown   
2          15  107.0            unknown  female       Unknown      unknown   
3          21   53.0            unknown  female       Unknown      unknown   
4          24   94.0            unknown    male       Unknown            3   

       region rural_urban cat_diabetes smoking  ...  \
0        East         NaN  DM unlikely       N  ...   
1         NaN         NaN  DM unlikely       N  ...   
2  North West         NaN         T2DM       N  ...   
3         NaN         NaN  DM unlikely       N  ...   
4         NaN         NaN  DM unlikely       N  ...   

   hospital_admissions_pre_6_9m  hospital_admissions_pre_9_12m  \
0                             0                              0   
1                             0                              0   
2 

## 3. Data Cleaning: Check Measures
Based on the link below and Joe's suggestions.  
https://github.com/Exeter-Diabetes/EHRBiomarkr/blob/main/data-raw/biomarker_acceptable_limits.yaml

In [33]:
df_features = clean_measure_values(df_features)

## 4. Check Missing Data

In [ ]:
#Exclude features with >80% missing values 
print ("Before excluding features with >80% missing values: ", len(df_features.columns))

missing_vals = ["", " ", "NA", "N/A", "na", "n/a", "None", "none", "NULL", "null", "Unknown", "unknown"]

df_features = df_features.replace(missing_vals, np.nan)

missing_prop = df_features.isnull().mean()

dropped_cols = missing_prop[missing_prop > 0.8].index.tolist()

df_features = df_features.loc[:, missing_prop <= 0.8]

print("Dropped columns:", dropped_cols)
print ("After excluding features with >80% missing values: ", len(df_features.columns))



Before excluding features with >80% missing values:  59
Dropped columns: ['imd_quintile', 'rural_urban']
After excluding features with >80% missing values:  57


## 5. Transform to One-hot-encoding

In [25]:
#Binary with missing values
binary_missing_cols = [
    col for col in df_features.columns
    if is_binary_column(df_features[col])
     and df_features[col].isna().any()
]

#Categorical columns
categorical_cols = [
    col for col in df_features.select_dtypes(include=["object", "category"]).columns
    if col != "patient_id"
]
df_features = pd.get_dummies(
    df_features,
    columns=binary_missing_cols+categorical_cols,
    dummy_na=True,
    drop_first=False,
    dtype=int
)
print("After one-hot encoding binary and categorical columns: ", len(df_features.columns))
#Check if there are still missing values
missing_after_ohe = df_features.isnull().mean()
missing_after_ohe = missing_after_ohe[missing_after_ohe > 0].sort_values(ascending=False)

print("Columns still containing missing values:")
print(missing_after_ohe)

After one-hot encoding binary and categorical columns:  81
Columns still containing missing values:
sysbp_value                   0.2026
diasbp_value                  0.2008
last_hdl_cholesterol_value    0.2004
last_hba1c_value              0.1972
last_cholesterol_value        0.1948
bmi_value                     0.1908
dtype: float64


## 6. Tain/Test Split

In [26]:
df_features["hf_outcome"] = df["hf_diagnosis_date"].notna().astype(int)
X = df_features.drop(columns=["patient_id"] + ["hf_outcome"])
y = df_features["hf_outcome"]
patient_ids = df_features["patient_id"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    patient_ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data: ", X_train.shape)
print("Test data: ", X_test.shape)
train_data = pd.concat(
    [
        id_train.reset_index(drop=True).rename("patient_id"),
        X_train.reset_index(drop=True),
        y_train.reset_index(drop=True).rename("hf_outcome"),
    ],
    axis=1
)

test_data = pd.concat(
    [
        id_test.reset_index(drop=True).rename("patient_id"),
        X_test.reset_index(drop=True),
        y_test.reset_index(drop=True).rename("hf_outcome"),
    ],
    axis=1
)

print("Final train data: ", train_data.shape)
print("Final test data: ", test_data.shape)


Training data:  (4000, 80)
Test data:  (1000, 80)
Final train data:  (4000, 82)
Final test data:  (1000, 82)


## 7. KNN Imputation

In [27]:
# impute_cols = [
#     col for col in X_train.columns
#     if X_train[col].dtype.kind in 'iufc' and X_train[col].isna().any() # signed/unsigned int, float, complex
# ]

# print("Columns to impute:")
# print(impute_cols)

# for col in impute_cols:
#     X_train[f"{col}_imputed"] = X_train[col].isna().astype(int)
#     X_test[f"{col}_imputed"] = X_test[col].isna().astype(int)

# imputer = KNNImputer(n_neighbors=5)

# X_train[impute_cols] = imputer.fit_transform(X_train[impute_cols])
# X_test[impute_cols] = imputer.transform(X_test[impute_cols])




# imputer = KNNImputer(n_neighbors=5)

# X_train = pd.DataFrame(
#     imputer.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )

# X_test = pd.DataFrame(
#     imputer.transform(X_test),
#     columns=X_test.columns,
#     index=X_test.index
# )
# #Check imputation
# print("Missing values in train after imputation: ", X_train.isnull().sum().sum())
# print("Missing values in test after imputation: ", X_test.isnull().sum().sum())

# #Re-attach patient_id
# train_data = pd.concat(
#     [
#         id_train.reset_index(drop=True).rename("patient_id"),
#         X_train.reset_index(drop=True),
#         y_train.reset_index(drop=True),
#     ],
#     axis=1
# )

# test_data = pd.concat(
#     [
#         id_test.reset_index(drop=True).rename("patient_id"),
#         X_test.reset_index(drop=True),
#         y_test.reset_index(drop=True),
#     ],
#     axis=1
# )

# print("Final train data: ", train_data.shape)
# print("Final test data: ", test_data.shape)


## 7. Checks

In [ ]:
print("Missing values in X_train: ", X_train.isnull().sum().sum())
print("Missing values in X_test: ", X_test.isnull().sum().sum())

missing_train = X_train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

missing_test = X_test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)

print("Columns with missing values in train:")
print(missing_train)

print("Columns with missing values in test:")
print(missing_test)

Missing values in X_train:  4765
Missing values in X_test:  1168
Columns with missing values in train:
last_hdl_cholesterol_value    816
sysbp_value                   807
last_hba1c_value              802
diasbp_value                  789
last_cholesterol_value        779
bmi_value                     772
dtype: int64
Columns with missing values in test:
diasbp_value                  215
sysbp_value                   206
last_cholesterol_value        195
last_hdl_cholesterol_value    186
last_hba1c_value              184
bmi_value                     182
dtype: int64


## 8. Save

In [ ]:

os.makedirs(os.path.dirname(TRAIN_DATA_PATH), exist_ok=True)
os.makedirs(os.path.dirname(TEST_DATA_PATH), exist_ok=True)

train_data.to_csv(TRAIN_DATA_PATH, index=False, compression="gzip")
test_data.to_csv(TEST_DATA_PATH, index=False, compression="gzip")

print("Saved train and test data")

Saved train and test data
